In [1]:
from pyspark.sql.functions import col, count, when

gold_path = "abfss://gold@olistdatalake2027.dfs.core.windows.net/"

display(mssparkutils.fs.ls(gold_path))

StatementMeta(olistsparkjob, 14, 2, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dc85c9c9-a9e5-429a-934b-f1e2b7791946)

In [4]:
from pyspark.sql.functions import col, count, when

df = spark.read.parquet(
    gold_path + "revenue_analysis/"
)

print("Total Records:", df.count())
print("Total Columns:", len(df.columns))

df.printSchema()
display(df.limit(10))

StatementMeta(olistsparkjob, 14, 5, Finished, Available, Finished, False)

Total Records: 24
Total Columns: 5
root
 |-- revenue_month: string (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- product_revenue: double (nullable = true)
 |-- freight_revenue: double (nullable = true)
 |-- total_revenue: double (nullable = true)



SynapseWidget(Synapse.DataFrame, b1376788-599b-472b-84a4-79d406a24cf8)

In [3]:
gold_path = "abfss://gold@olistdatalake2027.dfs.core.windows.net/"

files = mssparkutils.fs.ls(gold_path)

for file in files:
    print(file.name, file.path, file.isDir)

StatementMeta(olistsparkjob, 14, 4, Finished, Available, Finished, False)

category_analysis abfss://gold@olistdatalake2027.dfs.core.windows.net/category_analysis True
customer_analysis abfss://gold@olistdatalake2027.dfs.core.windows.net/customer_analysis True
delivery_analysis abfss://gold@olistdatalake2027.dfs.core.windows.net/delivery_analysis True
delivery_performance abfss://gold@olistdatalake2027.dfs.core.windows.net/delivery_performance True
delivery_summary abfss://gold@olistdatalake2027.dfs.core.windows.net/delivery_summary True
installment_analysis abfss://gold@olistdatalake2027.dfs.core.windows.net/installment_analysis True
monthly_delivery_analysis abfss://gold@olistdatalake2027.dfs.core.windows.net/monthly_delivery_analysis True
monthly_payment_analysis abfss://gold@olistdatalake2027.dfs.core.windows.net/monthly_payment_analysis True
payment_method_analysis abfss://gold@olistdatalake2027.dfs.core.windows.net/payment_method_analysis True
product_analysis abfss://gold@olistdatalake2027.dfs.core.windows.net/product_analysis True
revenue_analysis abf

In [5]:
print("Duplicate Records:",
      df.count() - df.dropDuplicates().count())

print("Null Values:")

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

StatementMeta(olistsparkjob, 14, 6, Finished, Available, Finished, False)

Duplicate Records: 0
Null Values:
+-------------+------------+---------------+---------------+-------------+
|revenue_month|total_orders|product_revenue|freight_revenue|total_revenue|
+-------------+------------+---------------+---------------+-------------+
|            0|           0|              0|              0|            0|
+-------------+------------+---------------+---------------+-------------+



In [6]:
datasets = [
    "category_analysis",
    "customer_analysis",
    "delivery_analysis",
    "delivery_performance",
    "delivery_summary",
    "installment_analysis",
    "monthly_delivery_analysis",
    "monthly_payment_analysis",
    "payment_method_analysis",
    "product_analysis",
    "revenue_analysis",
    "state_customer_analysis",
    "top_products"
]

for name in datasets:
    df = spark.read.parquet(gold_path + name + "/")
    print(name, ":", df.count(), "records")

StatementMeta(olistsparkjob, 14, 7, Finished, Available, Finished, False)

category_analysis : 74 records
customer_analysis : 99441 records
delivery_analysis : 99441 records
delivery_performance : 99441 records
delivery_summary : 3 records
installment_analysis : 23 records
monthly_delivery_analysis : 25 records
monthly_payment_analysis : 25 records
payment_method_analysis : 5 records
product_analysis : 32951 records
revenue_analysis : 24 records
state_customer_analysis : 27 records
top_products : 10 records


In [7]:
from pyspark.sql import Row

datasets = [
    "category_analysis",
    "customer_analysis",
    "delivery_analysis",
    "delivery_performance",
    "delivery_summary",
    "installment_analysis",
    "monthly_delivery_analysis",
    "monthly_payment_analysis",
    "payment_method_analysis",
    "product_analysis",
    "revenue_analysis",
    "state_customer_analysis",
    "top_products"
]

results = []

for name in datasets:
    try:
        df = spark.read.parquet(gold_path + name + "/")

        results.append(Row(
            dataset=name,
            records=df.count(),
            columns=len(df.columns),
            status="SUCCESS"
        ))

    except Exception as e:
        results.append(Row(
            dataset=name,
            records=0,
            columns=0,
            status="FAILED"
        ))

summary_df = spark.createDataFrame(results)

display(summary_df)

StatementMeta(olistsparkjob, 14, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d53f7d22-9812-49c7-8384-a32e1032b638)